# Chapter 2: Develop a Federated Application

Welcome to Chapter 2 of the course 5 minutes to Federated Learning with NVIDIA FLARE!

In this chapter, we focus on how to develop a federated application with NVIDIA FLARE's APIs and tools. We first introduce the high-level architecture of NVIDIA FLARE. Then, we walkthrough core user APIs of NVIDIA FLARE which allow scientists and developers to quickly build fundamental components of federated learning and adapt any centralized computation routine to federated paradigm. We illustrate FLARE APIs with a hands-on example of federated numerical computation using `numpy`. Finally, we put everything together and run the example using FL Simulator, a handy runtime tool that allows researchers to quickly test-run a federated job in a local development environment.

After this chapter, you will:
- Have a basic understanding of fundamental NVIDIA FLARE core user APIs
- Be able to quickly adapt a traditional centralized program to federated paradigm
- Be able to set up, configure and export a federated application for research and prototyping purpose
- Be able to run a federated application in a simulated environment 


# NVIDIA FLARE Architecture

The diagram below summarizes the high-level architecture of NVIDIA FLARE.

![NVFLARE Arch](../images/nvflare-arch.png)

Now let's look at this diagram in more details. In NVIDIA FLARE, a federated workflow is centered around the interaction between server-side "Controller" and client-side "Executors", through the concept of "Tasks". A "Task" is a piece of code that the server Controller assigns to client Executors. A task can be local training, local validation, or any other general routine, and the logic on how to perform a task is defined in an Executor. The interaction between server and client is defined and conceptualized as a "Federated Job". NVIDIA FLARE provides "Runtime" to run different jobs. 

Here are more details on these fundamental components:
- **Server-side Controller**: a Controller defines the server-side aggregation workflow and coordinates with client-side Executors through task assignments. NVIDIA FLARE provides server-side Controller classes with extensible APIs which allow developers to re-use classic federated workflows (scatter-and-gather, cyclic etc.) and classic optimization algorithms (FedAvg, FedOpt, etc.) or to efficiently implement customized workflows.
- **Client-side Executor**: an Executor implements details and logics on how to perform tasks received from the server-side Controller, whether the task is local training, or a general compute routine. In NVIDIA FLARE, client-side Executors can be created from existing centralized computation / training code, using Python APIs, allowing researchers to quickly and easily convert a traditional centralized program to a federated / distributed paradigm.
- **Federated Job**: NVIDIA FLARE provides `FedJob` class, an abstraction of server-client interaction, which allows users to set up and configure a federated workflow using Pythonic APIs. A `FedJob` can be exported and run by NVIDIA FLARE runtime.
- **Runtime**: NVIDIA FLARE provides different runtime backends to run federated jobs, either in a simulated environment to facilitate research and fast debugging & prototyping, or in a real-world scenario with capabilities to monitor and manage multiple jobs. We will focus on the FL Simulator in this notebook, which allows researchers to test-run federated jobs in a simulated environment on a local development PC, before real-world deployment.

Notice also the possibility of adding filters to task data and / or results, at any moment of the Controller & Executor interaction. This filtering mechanism, which we will briefly introduce in [Chapter 4](Chapter_4_Advanced_Topics_Use_Cases_and_Additional_Resources.ipynb), provides a flexible way to add security & privacy filters such as homomorphic encryption and differential privacy.


# Core User APIs

NVIDIA FLARE offers a rich set of APIs. In this notebook, we look at the core user APIs that allow you to adapt a centralized compute routine to federated paradigm in 5 minutes. These APIs can be summarized in 3 categories:
- **Server-side APIs**: these are essentially APIs for implementing server-side Controllers. In NVIDIA FLARE, implementing server-side Controllers is made easy with the [ModelController](https://nvflare.readthedocs.io/en/main/programming_guide/controllers/model_controller.html) class. This is a basic class allowing you to either re-use classic federated workflow provided by FLARE, or customize your own workflow.
- **Client-side APIs**: these are essentially APIs for implementing client-side Executors. NVIDIA FLARE provides [Client APIs](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html#client-api), a suite of carefully designed APIs that allow user to easily adapt any centralized compute to federated compute.
- **Job APIs**: these are essentially APIs to wrap server and client logics into a Federated Job, so that it can run under NVIDIA FLARE's runtime. More specifically, we will look at the [FedJob](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html) API.

In the following section, we will walk through the use of these core APIs with a toy example using `numpy`.


# Example: Federated Averaging

We will look at the **Hello FedAvg NumPy** example, taken from NVIDIA FLARE's [official example repository](https://github.com/NVIDIA/NVFlare/tree/109da964126c015d248718dbff7865206aa2ad6f/examples/hello-world/hello-fedavg-numpy). 

In this toy example, we will implement a simple distributed workflow, where starting from a numpy array `x`
- the server share `x` with the clients
- the clients independently update the value of the given array and send it back to the server
- the server aggregates the received arrays, compute their average and store it in `x`

This whole process will be repeated for a certain number of iterations / rounds.

We will show how you can easily implement this toy example using NVIDIA FLARE's **Server, Client and Job APIs.**


### Setup

Let's first copy the example to our notebook workspace

In [1]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/hello-fedavg-numpy ]; then \
    cp -r /NVFlare/examples/hello-world/hello-fedavg-numpy examples/hello-fedavg-numpy; fi
!tree examples/hello-fedavg-numpy

examples/hello-fedavg-numpy
├── README.md
├── fedavg_script_runner_hello-numpy.py
├── hello-fedavg-numpy_flare_api.ipynb
├── hello-fedavg-numpy_getting_started.ipynb
├── requirements.txt
└── src
    └── hello-numpy_fl.py

2 directories, 6 files


This will create an `examples` folder if it does not already exist. Inside the `examples` folder, you'll find the folder `hello-fedavg-numpy`, which has the content of the example that we will walkthrough together. 

You can see that we have many files in this example. We will ignore the `.ipynb` files, and focus solely on the two Python source code files: 
- [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)
- [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py).

To give you the best understanding of FLARE's APIs, we will walkthrough the two Python files in details, in the following order:
- First, we will look at the client-side implementation (in [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py)). As we've mentioned above, this implements Executors which define the local computation routine for each site. We start with client-side implementation because data scientists usually start with an existing centralized program, and convert it to be federated. We aim to show how easy and seamingless it is to do that using NVIDIA FLARE's Client APIs. 
- Then, we will look at the server-side implementation (first part of [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)) which creates the Controller to define the server side aggregation workflow.
- Finally, we will look at how to connect the server- and client-side implementations together, by wrapping them into a Federated Job. This is the rest of the file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py).


### Client-Side Implementation

Clients usually implement Executors which handle the computation routine / training logic that happens locally on each client site. In a traditional centralized computation paradigm, this is where one would write their core computation / training code. 

NVIDIA FLARE introduced the [**FLARE client API**](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html) in version 2.4, allowing users to simply convert traditional centralized workflows to federated ones through some convenient APIs. **Using client APIs alleviate the need of manually writing boilerplate code for creating and configuring Executors, making it extremely easy and seamingless to transition to a federated paradigm**.

To illustrate this, let's analyze the client code in [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py). If we ignore for a moment the NVIDIA FLARE related code, the main logic would be the following

```python
import copy
import numpy as np

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
    # This is our "mock" model that we aim to "train"
    input_model = ...

    # Get parameters from "mock" input_model: if empty, use default value.
    if input_model.params == {}:
        params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
    else:
        params = np.array(input_model.params["numpy_key"], dtype=np.float32)
        
    # mock training
    new_params = train(params)
    
    # mock evaluation
    metrics = evaluate(params)

if __name__ == "__main__":
    main()
```

You can see that this is nothing but a simple mock "training" & "evaluation" routine:
- The computation flow is defined in the `main()` function: the model we want to "train" in this example is an object named `input_model`. How to get the `input_model` object is not important for the moment. Our goal here is to "train" its parameters, represented by the `params["numpy_key"]` attribute, which is a `numpy` array. We first get a copy of the parameters `input_model.params["numpy_key"]` and assign it to an array named `params`. If `input_model.params["numpy_key"]` is empty, we use the default value `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]`.
- The `train()` function performs a mock "training" task: adding value `1` to each element of `params` array, and returning a new array `new_params`.
- The `evaluate()` function perform a mock "evaluation", by simply computing the mean value of the modified new array `new_params`.

**Now let's make this simple computation federated: we will show you step-by-step how easy it is to do that by adding NVIDIA FLARE's client APIs into the code above.**

- **Step 1: initialize FLARE**.

First, we need to import NVIDIA FLARE and initialize it
```python
import nvflare.client as flare

flare.init()
```

- **Step 2: receive model from server**.

The first code block in the original `main()` function above is about getting the `input_model`:
```python
# Get a "mock" model: input_model
input_model = ...
```
In traditional centralized compute, this would be the place for model loading or initialization. However, in a federated paradigm, a client does not need to worry about how to get a model, it simply *receives* a copy of the global model from the server. With NVIDIA FLARE, this is done using the `receive()` API. Therefore we can replace this first code block by:
```python
input_model = flare.receive()
```
This tells the client that it will, at some point, receive a model from the server. As of when and how, that is the server's concern. As of the format of the `input_model` received from the server, NVIDIA FLARE uses a standard class [**`flare.FLModel`**](https://nvflare.readthedocs.io/en/main/programming_guide/fl_model.html#flmodel), which is a dictionary-like seriablizable class. We will look at more details on `FLModel` later.

The rest of the code blocks in the `main()` function stays the same: we get a copy of the parameters `input_model.params["numpy_key"]` and assign it to an array `params`. If the parameters are empty, we use the default value. Then we perform our simple `train()` operation on the `params`: adding value `1` to the elements of the array, and get a new array `new_params`. Lastly, the `evaluate()` operation is performed, computing mean value of the new array `new_params`.

- **Step 3: send new model to server**.

There is something missing in the end of the `main()` function though. Remember that federated paradigm is a distributed and collaborative paradigm, where clients need to send the local computation results back to the server for aggregation, when local results are available. To do that, NVIDIA FLARE provides a convenient API `flare.send()`. As of the format of the results to be sent to the server, similar to `flare.receive()`, NVIDIA FLARE uses the standard `flare.FLModel` class. Adding the following in the end of the `main()` function, we send the computation results, a.k.a the modified `new_params` in our case, to the server:
```python
output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )
flare.send(output_model)
```
Notice that apart from the `params` argument which refers to the parameters of the model / computation results in a Python dictionary format, `flare.FLModel` has many other input arguments, such as `param_type`, `metrics`, `current_round`. Please refer to its [API documentation](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_common.abstract.fl_model.html#module-nvflare.app_common.abstract.fl_model) for more detailed explanation of these arguments.

- **Step 4: rinse and repeat**.

At this point, we are almost finished on the client side, but there is still one last element missing. Remember that a federated workflow is usually an **iterative** interaction process between server and clients. The server decides on how many iterations, or "rounds" of computations to be performed by each client: this needs to be handled on the client side. With NVIDIA FLARE, this is done by wrapping the client computation inside a `while` clause:
```python
while flare.is_running():
    # client computation code
    ...
```
As the name indicates, the computation wrapped inside the `while flare.is_running():` will be performed for as many iterations as required by the server.

**And that's it: we have converted a centralized computation code to federated client code in simply 4 steps!** 

We still need to create an Executor from this federated code, but you will see later that this is easy to do with some convenient APIs.

Putting all things together, we have the final federated client-side implementation:

```python
import copy
import numpy as np
import nvflare.client as flare # Import flare

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
   
    flare.init() # Initialization
    
    while flare.is_running(): # Run iteratively
        
        input_model = flare.receive() # Get copy of model from server

        if input_model.params == {}:
            params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
        else:
            params = np.array(input_model.params["numpy_key"], dtype=np.float32)

        # training
        new_params = train(params)
        # evaluation
        metrics = evaluate(params)

        # Send results to server
        output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )

        flare.send(output_model)

if __name__ == "__main__":
    main()
```

If you compare the federated implementation with the original one shown in the beginning of this section, you can notice that the difference is quite minimal: only additions of a few client APIs. Though the computation is this example is quite trivial, this type of adaptation can be done on practically any complex computations, as we will explore in later content of this course. 

To learn more about the client APIs, please refer to the [documentation here](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html).

### Server-Side Implementation

With the client-side implementation finished, let's now look at how to implement the server-side workflow. 

As we've explained above in NVIDIA FLARE's architecture, the server-side implementation consists of creating Controllers, which define server-side orchestration / aggregation workflow and the interaction logic with clients. Compared with traditional centralized compute, the server-side code is a completely new addition, therefore has to be implemented by users. Luckily, NVDIA FLARE provides an easy and extensible way to do it.

Let's look at the file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py). Below is the part of code that is related to server-side implementation:

```python
n_clients = 2
num_rounds = 3

persistor_id = job.to_server(NPModelPersistor(), "persistor")

# Define the controller workflow and send to server
controller = FedAvg(
    num_clients=n_clients,
    num_rounds=num_rounds,
    persistor_id=persistor_id,
)
```

You can see that the server code is simply initializing a `FedAvg` class provided by NVIDIA FLARE. That's because in this example, the server-side workflow is quite straight-forward: computing the average from results received from clients. This is the classic Federated Averaging workflow and we can directly leverage the available [FedAvg (Federated Averaging) Controller](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L18).

Let's discuss how the `FedAvg` Controller is used. It is instantiated with 3 arguments:
- `num_clients`: the number of clients to select for each training round. This number can actually be different from the actual total number of participating clients. It cannot be greater than the number of total clients, but it can be less, in which case, a random subset of `num_clients` clients will be selected for each round. Notice that, client sampling strategy is completely customizable in FLARE, as we will explain later on.
- `num_rounds`: the total number of rounds of aggregation to be performed. This also determines how many times the computation code wrapped inside the `while flare.is_running():` clause in client-side implementation will run.
- `persistor_id`: ID to a Persistor object.

A Persistor in NVIDIA FLARE is in charge of loading & serializing something; in our context, the global model on the server side. In this example, the global model is just a `numpy` array, therefore we can directly use the class [`NPModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/np/np_model_persistor.py#L39) provided by NVIDIA FLARE, which implements:
- Default array initialization: this is typically for the initial round of a federated workflow, where the server needs to create a default array and send it to the client.
- `save_model()`: for serializing the array to the filesystem as a `numpy` `.npy` file.
- `load_model()`: for loading a `.npy` file from the filesystem to a `numpy` array.

The ID of the created Persistor is passed to the `FedAvg` Controller, so that when the Controller needs to initialze the global model, saving the model to disk or loading a model from disk, it will call the corresponding function from the Persistor. FLARE provides many other default model Persistors, for instance the [`PTFileModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/pt/file_model_persistor.py#L36) for `Pytorch` models, [`TFModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/tf/model_persistor.py#L26) for `Tensorflow` models, etc. You also have the possibility to write your own model persistor by sub-classing the [ModelPersistor](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/abstract/model_persistor.py#L26) class. Also notice that, you do not have to use a Persistor object for model saving and loading: FLARE's base Controller class is flexible enough for you to implement your own model saving & loading logics, by simply overriding the corresponding functions.

That's it, simply calling the existing `FedAvg` Controller allows us to set up the server side Federated Averaging workflow for this toy example.

#### Understading the `ModelController`

Due to the simplicity of this example, there is no customization needed for the server side workflow. We can simply reuse the Federated Averaging workflow logic that is hidden inside the implementation of the `FedAvg` Controller. However, for more realistic and complex workflows, we need to have a deeper understanding of the fundamental aspects of Controllers: for instance, how to customize server aggregation logic, how to sample clients in a different way than Federated Averaging, etc. To better understand all these, let's look at how FLARE implements various Controllers leveraging the [ModelController](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L23) class. 

**The `ModelController` class is the fundational base Controller class, that allows developers to implement any custom server-side workflows.** 

From FLARE's architecture diagram, we know that a server side Controller needs to have the following capabilities:
- Orchestrate the overall server-client interaction workflow
- Initialize, load and save the global model
- Communication with available clients, send model and tasks to clients
- Perform aggregation of results returned by clients after task executions
- Update the global model

To offer a flexible and extensible way to implement these capabilities, the `ModelController` class provides the following key functionalities:
- [`run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L38): this is an asbtract function that must be overridden by all Controller classes that inherit from ModelController. This is the key function where the server-side workflow logic is orchestrated. 
- [`load_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L107) and [`save_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L115): these are methods implementing how to load and save the global model on the server-side, including initializing the first model. The global model is stored as an instance of the `FLModel` class, a format that FLARE uses for basically all model & data exchanges. In the `FedAvg` Controller, these methods actually call their corresponding function in the Persistor object that is passed to the `ModelController`. It is also perfectly ok to override these functions to accommodate your own customized model loading and saving, without relying on the concept of Persistors at all.
- [`sample_clients()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L126): this function queries the FLARE system and selects a list of participating clients for the current round. Noted that you can override this function to implement your own custom client sampling strategy.
- [`send_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L74) and [`send_model_and_wait()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L42): these functions send copies of the global `FLModel` to the clients, and expects clients to perform local tasks (training, validation etc.) and send the results back to server. Client results are also wrapped as `FLModel` instances to leverage the uniform data exchange format. Inside these function, low-level communications between server and client are implemented.
- Besides these basic common APIs, you also have the flexibility to implement any workflow related custom functions, and orchestrate them in the `run()` function. For instance, you will probably need to implement custom algorithms on how to aggregate the global model from client local results, and how to update the global model on the server side based on aggregated result. In the `FedAvg` Controller, this is done via custom [`aggregate()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115) and [`update_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L148) functions, as we will see in details later.

In practice, developers typically do not need to worry about low-level communications which are hidden inside `send_model()` and `send_model_and_wait()`'s implementations. All they need to override is workflow-related logic:
- the `run()` function which orchestrates the overall workflow
- model loading & saving: via `load_model()` and `save_model()` functions
- clients sampling strategy: via `sample_clients()`
- server-side model aggregation algorithm: for instance `aggregate()` and `update_model()` in `FedAvg` Controller
- and any other custom routines needed by the workflow

This design of `ModelController` separates low-level communication code from workflow-related code, offering a flexible and easy way to customize any server-side workflow. Also, the use of `FLModel` format for all model & data exchanges makes it easier to understand and customize the data flow.

#### Implementing Federated Averaging Controller Using `ModelController`

To give you a concrete example, let's look back at how Federated Averaging ([FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L18) and [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L29) classes) is implemented under the hood, leveraging `ModelController`. The `run()` method is overriden in the [FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L34) class:
 ```python
    def run(self) -> None:
        model = self.load_model()
        model.start_round = self.start_round
        model.total_rounds = self.num_rounds

        for self.current_round in range(self.start_round, self.start_round + self.num_rounds):
            model.current_round = self.current_round
            clients = self.sample_clients(self.num_clients)
            results = self.send_model_and_wait(targets=clients, data=model)
            aggregate_results = self.aggregate(results, aggregate_fn=self.aggregate_fn)
            model = self.update_model(model, aggregate_results)
            self.save_model(model)
```
Inside the `run()` function, a classic Federated Averaging workflow is orchestrated:
- An initial model is loaded with [`load_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L107) method. By default, this internally calls the `load_model()` function from the Persistor.
- For each round, we get a list of active clients via [`sample_clients()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L126), and send them each a copy of the current model for local training via [`send_model_and_wait()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L42). Here we don't need to care about the low-level server-client communication routines, these are handled by `send_model_and_wait()` internally.
- When local clients return results for each round, we perform model aggregation via [`aggregate()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115) function. The aggregation function, implemented in class [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115), performs a simple weighted average of all the clients' local training results.
- When the aggregated result is ready, we update the global model via [`update_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/utils/fl_model_utils.py#L222) function. The model update function is implemented as a utility function in class [FLModelUtils](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/utils/fl_model_utils.py#L44).
- Finally, we save the global model for each round, via the [`save_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L115) function.

As we can see, since `ModelController` already handles the low-level communications, and hide them from workflow-related logics, it allows developers to focus only on workflow-related implementation when implementing server-side code.

**And that's it: we've walked-through the implementation of server-side Controllers leveraging the flexible ModelController class!**

In practice, the [FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py) and [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py) classes, built on top of `ModelController`, can be leveraged to implement most of the centralized federated workflows, with the flexibility to easily customize server-side aggregation and model update functions. Besides `FedAvg` Controller, NVIDIA FLARE also provides many other reference server-side workflow implementations, for instance, [cyclic workflow](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cyclic.py), [Scaffold](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/scaffold.py), [cross-site evaluation](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cross_site_eval.py) etc. 

You can read the [documentation here](https://nvflare.readthedocs.io/en/main/programming_guide/controllers/model_controller.html) to learn more about the `ModelController` with more examples.


### Putting Everything into a Federated Job

Now we have the client- and server-side implementations, all that remains to be done is to put everything together into a Federated Job, so that it can be run by NVIDIA FLARE's runtime. 

This can be easily acheived using the [FedJob API](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html). 

Let's look at the whole content of [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py) (order of lines modified to facilitate explanation):

```python
from nvflare import FedJob
from nvflare.app_common.np.np_model_persistor import NPModelPersistor
from nvflare.app_common.widgets.intime_model_selector import IntimeModelSelector
from nvflare.app_common.workflows.fedavg import FedAvg
from nvflare.job_config.script_runner import FrameworkType, ScriptRunner

if __name__ == "__main__":

    job = FedJob(name="hello-fedavg-numpy")

    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")

    job.to(IntimeModelSelector(key_metric="accuracy"), "server")

    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")

    job.export_job("/tmp/nvflare/jobs/job_config")
    job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```

Let's analyze this file. First, we import the `FedJob` class and define a `FedJob` object:
```python
    from nvflare import FedJob
    ...
    job = FedJob(name="hello-fedavg-numpy")
```
The [`FebJob`](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html) class aims to model the federated workflow and server-client interactions, and it offers APIs to Pythonically define and configure federated jobs.

The next couple of lines are related to server-side Controller, as we've already seen in the server-side implementation section:
```python
    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")
```
Here we set the number of participating clients for each round to be 2, and we set up for 3 rounds of "training". 

We also see the usage of `FedJob.to` API: `FedJob.to` sends an object to a specific target, where the target can be a client or the server. In FLARE, the runtime system requires that each instantiated component to be sent to its corresponding participants: any components that run on the server side, e.g., Controllers and Persistors, should be sent to server side, while any components that run on the client side, e.g., Executors, should be sent to the client side.

The API call: `job.to_server(...)` is equivalent to `job.to(..., "server")`. Therefore we can interpret the above usage as follows:
- `job.to_server(NPModelPersistor(), "persistor")`: sends an instance of `NPModelPersistor` to the server, give the instance an ID of "persistor"
- `job.to(controller, "server")`: send the FedAvg Controller to the server. This is equivalent to `job.to_server(controller)`

The next line:
```python
job.to(IntimeModelSelector(key_metric="accuracy"), "server")
```
sends another component, `IntimeModelSelector` to the server. We would not go into details of this component in this course, but all you need to know is that this component allows the server to select & save the best global model during federated training rounds, based on specific metric, which in this case is the "accuracy".  This is possible since each client sends an `FLModel` object to the server, which has an attribute `metrics` that the `IntimeModelSelector` can refer to. For more details, please check out the [documentation](https://nvflare.readthedocs.io/en/main/programming_guide/component_configuration.html#component-configuration-and-event-handling). In this example, the model selection based on a "mock" evaluation accuracy does not really make sense, but we will see its better use-case later in this course with other more realistic examples.

Next, we have a couple of lines that refer to the client-side code:
```python
    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")
```
Here, you can think of it as: we are essentially turning our client-side Python script [`examples/hello-fedavg-numpy/src/hello-numpy_fl.py`](examples/hello-fedavg-numpy/src/hello-numpy_fl.py) into an Executor using the [`ScriptRunner`](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html#scriptrunner) API, and send it to each of the clients using `job.to` API (in reality, it is a bit more complicated, feel free to refer to [`ScriptRunner`'s implementation](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/script_runner.py#L30) for a deeper analysis). All the magic here is done inside `ScriptRunner`, which alleviates the need to manually write custom Executors. Instead, developers can start from a centralized training script, adapt it to a federated client-side local training script by adding FLARE client APIs, and then convert it to an Executor using `ScriptRunner`.

Finally, the definition & configuration of a Federated Job is complete, we can either run it directly, using for instance the FL Simulator:
```python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```

Or export it to a local folder and run it later using different runtime backends:
```python
job.export_job("/tmp/nvflare/jobs/job_config")
```

Let's export the job to a local folder. You will need to uncomment the line with `job.export_job`, and comment the line with `job.simulator_run` in [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)

Then, run the current script:

In [2]:
!cd examples/hello-fedavg-numpy && python fedavg_script_runner_hello-numpy.py

This will export the job under `/tmp/nvflare/jobs/job_config`, in a folder with name `hello-fedavg-numpy`, the same name that we used to intialize the `FedJob` class:

In [3]:
!tree /tmp/nvflare/jobs/job_config/hello-fedavg-numpy

/tmp/nvflare/jobs/job_config/hello-fedavg-numpy
├── app_server
│   └── config
│       └── config_fed_server.json
├── app_site-1
│   ├── config
│   │   └── config_fed_client.json
│   └── custom
│       └── src
│           └── hello-numpy_fl.py
├── app_site-2
│   ├── config
│   │   └── config_fed_client.json
│   └── custom
│       └── src
│           └── hello-numpy_fl.py
└── meta.json

11 directories, 6 files


**This is the typical folder structure of exported federated applications in FLARE**:

- A meta configuration file: `meta.json`

This file is used to capture the federation's meta information, including name, minimum number of clients, mandatory clients and which participants should this application be deployed to etc. This file is automatically generated by the `FedJob` API, and there are many ways to configure it using the `FedJob` API. For more details on this, read the documentation [here](https://nvflare.readthedocs.io/en/main/real_world_fl/job.html).

- Application folders: `app_server`, `app_site-1`, `app_site-2`, ...

These are folders representing applications that will be deployed to the corresponding server and clients at the start of the runtime. 

In this example, we defined 1 server and 2 clients. The `app_server` folder contains the application that will be deployed to the server and `app_site-1` and `app_site-2` folders are the applications that will be deployed to the 2 clients. Each application usually contains configurations in the `config` folder, and custom codes in the `custom` folder:
  - Server and client's configuration files capture their corresponding configurations defined using `FedJob` API. For instance, if `FedJob.to` was used to send a component to the client, you will find a field in the client's `config_fed_client.json` file describing the same component.
  - Custom codes are additional runtime codes required to run the application. For both clients, we can see that the `custom` folder includes the [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py) script that we've created together using client API during the "Client-side Implementation" section. For the server, there is no `custom` folder since the server just calls the `FedAvg` controller provided by FLARE, and there is no additional custom codes on the server side. But imagine if you customized a new controller using the `ModelController` base class and sent it to the server using `FedJob.to`, codes related to the new controller would be copied in the `custom` folder of the server's application.

For more details regarding server and client applications, refer to the documentation [here](https://nvflare.readthedocs.io/en/main/real_world_fl/application.html).

**And we are all done: we have implemented and configured our first federated application using NVIDIA FLARE!**

Now, we will run this federated application using FLARE's FL Simulator runtime together and observe the result.

### Running the Application with FL Simulator

The [FL Simulator](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_cli/fl_simulator.html) is a handy runtime designed for researchers to quickly test-run a federated application in a local development environment. It provides a convenient tool for fast prototyping and software / algorithm debugging before deployment in production. 

We have already seen the usage of `job.simulator_run()` method to run an application using FL Simulator in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py). The first argument to [`job.simulator_run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) is a directory to run the federated application and save results. The function also accepts multiple other arguments, including total number of clients, number of threads, GPU index etc. See [here](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) for details.

Noted that FL Simulator is designed for convenience, therefore it does not include necessary security features for real-world deployment. We will cover real-world deployment in later chapters of this course.

Let's uncomment the line: 
``` python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
``` 
in script [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py), and execute it. This will runs the federated numpy application with FL Simulator using `FedJob`'s API:

In [ ]:
!cd examples/hello-fedavg-numpy && python3 fedavg_script_runner_hello-numpy.py

There is quite a lot of console output, but you can essentially pick up a couple of prints, indicating that the initial `numpy` array `[[1,2,3], [4,5,6], [7,8,9]]` is incremented by value 1 to each of its elements after each round, and becoming `[[4,5,6], [7,8,9], [10,11,12]]` after a total of 3 rounds.


Let's also check the final best global model aggregated on the server-side. In this toy example, each client is simply adding 1 to the same fixed array for 3 rounds, so the average array aggregated at the server side should have the same value as the array returned from each client. Then, the server selects the best model based on the mock "accuracy", which is simply the mean value of the array. Therefore, we should expect the final best global model to contain the array with the largest mean value, i.e., `[[4,5,6], [7,8,9], [10,11,12]]`. Let's check if it is the case with the following code.

In [5]:
import numpy as np

server_model_path = "/tmp/nvflare/jobs/workdir/server/simulate_job/models/server.npy"
best_model = np.load(server_model_path)

print(best_model)

[[ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]


We can see that, the final best global model indeed conatins the array with value `[[4,5,6], [7,8,9], [10,11,12]]`, as expected.

Alternatively, we can run an exported Federated Job using NVIDIA FLARE's CLI tool for FL Simulator: `nvflare simulator`. Let's have a look at the CLI's help page, by running the following command:

In [6]:
!nvflare simulator -h

usage: nvflare simulator [-h] [-w WORKSPACE] [-n N_CLIENTS] [-c CLIENTS]
                         [-t THREADS] [-gpu GPU] [-l LOG_CONFIG]
                         [-m MAX_CLIENTS] [--end_run_for_all]
                         job_folder

positional arguments:
  job_folder

options:
  -h, --help            show this help message and exit
  -w WORKSPACE, --workspace WORKSPACE
                        WORKSPACE folder
  -n N_CLIENTS, --n_clients N_CLIENTS
                        number of clients
  -c CLIENTS, --clients CLIENTS
                        client names list
  -t THREADS, --threads THREADS
                        number of parallel running clients
  -gpu GPU, --gpu GPU   list of GPU Device Ids, comma separated
  -l LOG_CONFIG, --log_config LOG_CONFIG
                        log config file path
  -m MAX_CLIENTS, --max_clients MAX_CLIENTS
                        max number of clients
  --end_run_for_all     flag to indicate if running END_RUN event for all
                        

Similar to [`job.simulator_run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496), the CLI tool also provides similar configuration options. We won't be changing much of the options here, so let's just proceed with the default values, apart from setting the workspace to `/tmp/nvflare/jobs/workdir-cli`. Feel free to experiment with different options of the CLI on your own.

Let's run the previously exported application `/tmp/nvflare/jobs/job_config/hello-fedavg-numpy`:

In [ ]:
!nvflare simulator /tmp/nvflare/jobs/job_config/hello-fedavg-numpy -w /tmp/nvflare/jobs/workdir-cli

After the run is complete, we should expect exactly the same result:

In [8]:
np.load('/tmp/nvflare/jobs/workdir-cli/server/simulate_job/models/server.npy')

array([[ 4.,  5.,  6.],
       [ 7.,  8.,  9.],
       [10., 11., 12.]], dtype=float32)

As you can see, we get the same result as expected.

To finish up, let's remove temporary files generated by FLARE during the executions:

In [ ]:
!rm -rf /tmp/nvflare/

**That's it, we have successfully set up, configured a federated application, and run it using FLARE Simulator!**


# Recap

Let us recap what we've learned in this chapter:
- We first had an overview of NVIDIA FLARE's architecture, and learned about the concept of server Controllers, client Executors and Federated Job
- We then had an introduction to FLARE's core user APIs, which allows for quick implementations of Controllers, Executors and Federated Job configuration.
- We then walked through together a hands-on example using `numpy` to develop a toy federated application. In the example, we learned:
  - How to use FLARE's Client APIs to convert a centralized compute code to Federated
  - How to leverage `ModelController` class to implement Federated Averaging, as well as to customize any federated server workflow
  - How to wrap everything up into a Federated Job using the FedJob APIs
- Finally, we run together the federated `numpy` application using FLARE's FL Simulator runtime, and observed result that makes sense

Below is a useful diagram to recap the 3-step process to develop a federated application in FLARE:

![App Dev](../images/app-dev.png)


Now you've learned how to develop a federated application and run it in a simulated environment. You might wonder how we should run a federated project in real life. This is what we will cover in the [next chapter](Chapter_3_Provision_Run_and_Monitor_Federated_Project.ipynb). But before continuing to the next chapter, here are some exercises to help you grasp the essential of this chapter!

# Exercise 1

Change the client side "training" routine of the Hello FedAvg NumPy example: instead of adding value 1 to the elements of the parameter array, let's compute the square-root of each element of the array. Try exporting the new modified job, and run to inspect the results.

## Solution to Exercise 1

Modify the `train()` function in [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py), from:

```python
def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1
```

to:

```python
def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return np.sqrt(output_arr)
```

Then run the job with the command below. Don't forget to uncomment line `job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")` if you have previously commented it. You can of course export the job and run it with `nvflare simulator`, which gives you the same result.

In [ ]:
!cd examples/hello-fedavg-numpy && python3 fedavg_script_runner_hello-numpy.py

You should expect the following result for the final global model.

In [ ]:
np.load("/tmp/nvflare/jobs/workdir/server/simulate_job/models/server.npy")

# Exercise 2

Change the server side workflow of the Hello FedAvg NumPy example: from Federated Averaging (FedAvg) to Cyclic Weight Transfer (CWT) workflow. In CWT workflow, for each round, the server passes model parameters from the previous client to the next one for iterative training, until it reaches the last client. Read [here](https://github.com/NVIDIA/NVFlare/tree/main/examples/hello-world/hello-cyclic) to learn more about CWT workflow: but be careful to not look into the code there immediately, it contains the solution for this exercise :p

## Solution to Exercise 2

FLARE provides an existing implementation for Cyclic Weight Weight Transfer Controller, named [`Cyclic`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cyclic.py#L18). So we just need to replace the `FedAvg` Controller by the `Cyclic` Controller in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py):

First import the [`Cyclic`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cyclic.py#L18) Controller:
```python
from nvflare.app_common.workflows.cyclic import Cyclic
```

Then change
```python
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
```

to
```python
    controller = Cyclic(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
```

The `Cyclic` Controller is also implemented by sub-classing from the `ModelController` base class, so we do not need to change any input parameters, just the name of the Controller.

If we look into the implementation of [`Cyclic`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cyclic.py#L18), we can see that it's essentially just overriding the `run()` function of the base `ModelController` class:

```python
    def run(self) -> None:
        self.info("Start Cyclic.")

        model = self.load_model()
        model.start_round = self.start_round
        model.total_rounds = self.num_rounds

        for self.current_round in range(self.start_round, self.start_round + self.num_rounds):
            self.info(f"Round {self.current_round} started.")
            model.current_round = self.current_round

            clients = self.sample_clients(self.num_clients)

            for client in clients:
                result = self.send_model_and_wait(targets=[client], data=model)[0]
                model.params, model.meta = result.params, result.meta

            self.save_model(model)

        self.info("Finished Cyclic.")
```

Run the job with the command below:

In [ ]:
!cd examples/hello-fedavg-numpy && python3 fedavg_script_runner_hello-numpy.py

You should expect the following result for the final global model.

In [ ]:
np.load("/tmp/nvflare/jobs/workdir/server/simulate_job/models/server.npy")